# AIFS vs ERA5 RMSE Visualization (Precomputed Outputs)

This notebook **only loads precomputed RMSE outputs** produced by the pipeline.
No recomputation is performed. Inputs expected:
- `pairs_manifest_<variable>.csv`
- `rmse_by_step_<variable>.csv`
- `rmse_by_step_by_day_<variable>.csv`
- `rmse_map_week_<variable>.nc`


In [1]:
# Config: set paths to outputs + raster resources
from pathlib import Path

BASE_DIR = Path('.')  # update if needed
OUTPUT_DIR = BASE_DIR / 'outputs'  # folder with precomputed outputs

# Raster inputs (update paths)
KOPPEN_RASTER = Path('data_access/data/static//koppen_geiger_0p1.tif')
OROG_RASTER = Path('/data_access/data/static//orography.tif')  # or grib/nc if you have it

# Variable keys (as used in file names)
VARIABLES = ['2t', 't500']


In [2]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import rasterio
from rasterio.transform import rowcol
from matplotlib.colors import ListedColormap

EUROPE_EXTENT = (-5, 25, 43, 58)  # lon_min, lon_max, lat_min, lat_max

# --- Data loaders ---
def load_rmse_map(variable: str) -> xr.DataArray:
    path = OUTPUT_DIR / f'rmse_map_week_{variable}.nc'
    ds = xr.open_dataset(path)
    # assume single variable in file
    if len(ds.data_vars) == 1:
        return next(iter(ds.data_vars.values()))
    # fallback: try common name
    return ds[list(ds.data_vars)[0]]

def load_rmse_by_step(variable: str) -> pd.DataFrame:
    path = OUTPUT_DIR / f'rmse_by_step_{variable}.csv'
    return pd.read_csv(path)

def load_rmse_by_day(variable: str) -> pd.DataFrame:
    path = OUTPUT_DIR / f'rmse_by_step_by_day_{variable}.csv'
    return pd.read_csv(path)

# --- Map helpers ---
def setup_map_ax():
    fig = plt.figure(figsize=(10, 6))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(EUROPE_EXTENT, crs=ccrs.PlateCarree())
    return fig, ax

def add_koppen_background(ax, koppen_path: Path):
    # Load raster
    src = rasterio.open(koppen_path)
    data = src.read(1)
    transform = src.transform

    # Simple class mapping (code -> label)
    koppen_lookup = {
        1: 'Af', 2: 'Am', 3: 'Aw', 4: 'BWh', 5: 'BWk', 6: 'BSh', 7: 'BSk',
        8: 'Csa', 9: 'Csb', 10: 'Csc', 11: 'Cwa', 12: 'Cwb', 13: 'Cwc',
        14: 'Cfa', 15: 'Cfb', 16: 'Cfc', 17: 'Dsa', 18: 'Dsb', 19: 'Dsc', 20: 'Dsd',
        21: 'Dwa', 22: 'Dwb', 23: 'Dwc', 24: 'Dwd', 25: 'Dfa', 26: 'Dfb',
        27: 'Dfc', 28: 'Dfd', 29: 'ET', 30: 'EF'
    }

    # Crop raster to Europe extent
    lon_min, lon_max, lat_min, lat_max = EUROPE_EXTENT
    row_min, col_min = rowcol(transform, lon_min, lat_max)
    row_max, col_max = rowcol(transform, lon_max, lat_min)
    row_min, row_max = sorted([row_min, row_max])
    col_min, col_max = sorted([col_min, col_max])
    data_crop = data[row_min:row_max+1, col_min:col_max+1]

    # Identify classes present in the extent
    present_codes = sorted({int(c) for c in np.unique(data_crop) if int(c) in koppen_lookup})
    labels = [koppen_lookup[c] for c in present_codes]

    # Build a discrete colormap for only present classes
    colors = plt.cm.tab20.colors
    cmap = ListedColormap([colors[i % len(colors)] for i in range(len(present_codes))])
    code_to_index = {code: i for i, code in enumerate(present_codes)}
    indexed = np.vectorize(lambda v: code_to_index.get(int(v), np.nan))(data_crop)

    # Plot
    ax.imshow(
        indexed,
        origin='upper',
        extent=[lon_min, lon_max, lat_min, lat_max],
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        alpha=0.6,
        zorder=0,
    )

    # Legend with only present classes
    handles = [plt.Line2D([0], [0], marker='s', linestyle='', color=cmap(i), label=lbl)
               for i, lbl in enumerate(labels)]
    ax.legend(handles=handles, title='Köppen–Geiger', loc='lower left', fontsize=8)

def add_orography_overlay(ax, orog_path: Path):
    # Placeholder: load orography raster (GeoTIFF) and overlay semi-transparent shading
    src = rasterio.open(orog_path)
    data = src.read(1)
    transform = src.transform

    lon_min, lon_max, lat_min, lat_max = EUROPE_EXTENT
    row_min, col_min = rowcol(transform, lon_min, lat_max)
    row_max, col_max = rowcol(transform, lon_max, lat_min)
    row_min, row_max = sorted([row_min, row_max])
    col_min, col_max = sorted([col_min, col_max])
    data_crop = data[row_min:row_max+1, col_min:col_max+1]

    ax.imshow(
        data_crop,
        origin='upper',
        extent=[lon_min, lon_max, lat_min, lat_max],
        transform=ccrs.PlateCarree(),
        cmap='Greys',
        alpha=0.25,
        zorder=1,
    )

def add_gridlines(ax):
    gl = ax.gridlines(draw_labels=True, linestyle='--', linewidth=0.5, alpha=0.6)
    gl.top_labels = False
    gl.right_labels = False
    return gl

def add_colorbars(fig, ax, mappable, label: str):
    cbar = fig.colorbar(mappable, ax=ax, orientation='vertical', shrink=0.8)
    cbar.set_label(label)
    return cbar


ModuleNotFoundError: No module named 'rasterio'

In [ ]:
# Sanity checks for rmse_map
for var in VARIABLES:
    da = load_rmse_map(var)
    print(f'Variable: {var}')
    print('  dims:', da.dims)
    print('  shape:', da.shape)
    if 'latitude' in da.coords:
        print('  lat range:', float(da.latitude.min()), 'to', float(da.latitude.max()))
    if 'longitude' in da.coords:
        print('  lon range:', float(da.longitude.min()), 'to', float(da.longitude.max()))
    print('  min/max:', float(da.min()), float(da.max()))
    nan_frac = float(np.isnan(da.values).sum()) / da.size
    print('  NaN fraction:', nan_frac)
    print('-' * 40)
